In [3]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import os
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

# Initialize VADER
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

# Load News
news_path = os.path.join('..', 'data', 'raw', 'newsData', 'raw_analyst_ratings.csv')
news_df = pd.read_csv(news_path)

# Load Stocks
stock_path = os.path.join('..', 'data', 'raw', 'yfinance_data', 'Data', 'AAPL.csv')
stock_df = pd.read_csv(stock_path)

# Prepare Stock DataFrame
stock_df['Date'] = pd.to_datetime(stock_df['Date']).dt.tz_localize(None) # Remove timezone for clean merge
stock_df.set_index('Date', inplace=True)
stock_df['daily_return'] = stock_df['Close'].pct_change() * 100


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/sifen/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
print("Calculating sentiment scores... this may take a minute...")

# Optimization: Using a list comprehension is often faster than .apply() for VADER
news_df['sentiment'] = [sia.polarity_scores(str(h))['compound'] for h in news_df['headline']]

print("Sentiment calculation complete.")

Calculating sentiment scores... this may take a minute...


In [ ]:
# 1. Convert news date to datetime and strip timezone
news_df['date'] = pd.to_datetime(news_df['date'], utc=True).dt.tz_localize(None)

# 2. Function to move weekend news to Monday
def align_to_trading_day(ts):
    if ts.weekday() == 5: # Saturday
        return ts + pd.Timedelta(days=2)
    elif ts.weekday() == 6: # Sunday
        return ts + pd.Timedelta(days=1)
    return ts

# Apply alignment and strip to midnight for merging
news_df['trading_date'] = news_df['date'].apply(align_to_trading_day).dt.normalize()

# 3. Aggregate sentiment by day
daily_sentiment = news_df.groupby('trading_date')['sentiment'].mean()

In [ ]:
# Ensure indices are named the same for a perfect merge
daily_sentiment.index.name = 'Date'

# Merge
merged_df = pd.merge(daily_sentiment, stock_df[['daily_return']], 
                     left_index=True, right_index=True, how='inner')

# Calculate Correlation
correlation = merged_df['sentiment'].corr(merged_df['daily_return'])

print(f"Merged Data Points: {len(merged_df)}")
print(f"Pearson Correlation: {correlation:.4f}")

# Quick Visualization
plt.figure(figsize=(10, 6))
sns.scatterplot(data=merged_df, x='sentiment', y='daily_return', alpha=0.5)
plt.title(f'Sentiment vs Daily Returns (Correlation: {correlation:.4f})')
plt.show()